# 13 · Final stacking and the best submissions

Combines every saved model into the final files. Each combiner is scored with **nested cross-validation**: the combination weights are learned on four fifths of the patients and scored on the remaining fifth, so the score is not self-graded.

**Needs:** everything from notebooks 05-12. **Writes:** `submission_best.csv`, `submission_v3_additive_stack.csv`, `submission_v3_hillclimb.csv`, `submission_v4_stack.csv`. **Time:** about 20 minutes.

| File | What it is | Nested CV AUC |
|---|---|---|
| `submission.csv` | LightGBM + logistic blend (notebook 05) | 0.7526 |
| `submission_best.csv` | stack of boosted trees + logistic | 0.7527 |
| `submission_v3_additive_stack.csv` | stack that also uses the GAM / EBM models | **0.7534** (recommended) |
| `submission_v3_hillclimb.csv` | same pool, non-negative weights only | 0.7531 (safer) |
| `submission_v4_stack.csv` | v3 plus the notebook-12 models | 0.7534 (no real change) |

The gain from v3 over the original blend is about +0.0008 AUC (bootstrap: better in 98% of resamples). Around 0.754 looks like the practical ceiling for this dataset.

**Setup.** Install the libraries once (uncomment), then set the two paths in the next cell.

In [ ]:
# %pip install -q lightgbm xgboost scikit-learn pandas pyarrow scipy

In [ ]:
import os, sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# ---- EDIT THESE TWO PATHS --------------------------------------------------
DATA = os.environ.get('KAGGLE_DATA', 'data')   # folder holding the unzipped competition CSVs (train.csv, test.csv, patients.csv, ...)
WORK = os.environ.get('KAGGLE_WORK', 'work')   # folder where features, predictions and submissions are saved (shared by all notebooks)
# -----------------------------------------------------------------------------
Path(WORK).mkdir(parents=True, exist_ok=True)
print('DATA ->', Path(DATA).resolve()); print('WORK ->', Path(WORK).resolve())

### 13.1 Stack v1 → `submission_best.csv`
(the pool at that point: enriched-feature models + older models)

In [ ]:
import numpy as np, pandas as pd, glob, os
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
W=WORK; te=pd.read_csv(f'{DATA}/test.csv'); ss=pd.read_csv(f'{DATA}/sample_submission.csv')
X=pd.read_parquet(f'{W}/features2.parquet',columns=['encounter_id']); y=X.merge(pd.read_csv(f'{DATA}/train.csv'),on='encounter_id').readmit_30d.values
lg=lambda p:np.log(np.clip(p,1e-9,1-1e-9)/(1-np.clip(p,1e-9,1-1e-9)))
cur=pd.read_csv(f'{W}/submission.csv'); M={}
for f in sorted(glob.glob(f'{W}/n_*_oof.npy')): k=os.path.basename(f)[2:-8]; M['new_'+k]=(np.load(f),np.load(f.replace('_oof','_test')))
M['old_lgb_lr_blend']=(np.load(f'{W}/final_oof.npy'),cur.readmit_30d.values)
M['old_xgb']=(np.load(f'{W}/xgb_oof_single.npy'),np.load(f'{W}/xgb_test_single.npy')); M['old_cat']=(np.load(f'{W}/b_cat_oof.npy'),np.load(f'{W}/b_cat_test.npy'))
for k in ['mlp']: M['old_'+k]=(np.load(f'{W}/b_{k}_oof.npy'),np.load(f'{W}/b_{k}_test.npy'))
M['old_rf']=(np.load(f'{W}/rf_oof.npy'),np.load(f'{W}/rf_test.npy')); M['old_nb']=(np.load(f'{W}/nb_oof.npy'),np.load(f'{W}/nb_test.npy'))
names=list(M); O=np.column_stack([lg(M[k][0]) for k in names]); Ts=np.column_stack([lg(M[k][1]) for k in names]); mu,sd=O.mean(0),O.std(0); O=(O-mu)/sd; Ts=(Ts-mu)/sd
print('Individual OOF AUC:'); [print(f'  {k:18s} {roc_auc_score(y,O[:,i]):.5f}') for i,k in enumerate(names)]
def stack(idx,C):
    nest=np.zeros(len(y))
    for a,b in StratifiedKFold(5,shuffle=True,random_state=42).split(O,y): nest[b]=LogisticRegression(C=C).fit(O[a][:,idx],y[a]).decision_function(O[b][:,idx])
    return roc_auc_score(y,nest)
allidx=list(range(len(names))); gb=[i for i,k in enumerate(names) if k.startswith('new_') or k in('old_lgb_lr_blend','old_xgb','old_cat')]
print('\nNested-CV stacking AUC:')
for lbl,idx in [('all models',allidx),('GBMs/LR only',gb),('old best blend alone',[names.index('old_lgb_lr_blend')])]:
    for C in [0.01,0.1,1]: print(f'  {lbl:22s} C={C}: {stack(idx,C):.5f}')
best=(stack(gb,0.01),'gb',gb,0.01); print('USING stack:',best[0],best[1],'C=',best[3])
idx,C=best[2],best[3]; st=LogisticRegression(C=C).fit(O[:,idx],y); print('weights:',{names[i]:round(w,3) for i,w in zip(idx,st.coef_[0])})
p=st.predict_proba(Ts[:,idx])[:,1]; sub=pd.DataFrame({'encounter_id':te.encounter_id,'readmit_30d':np.round(p,6)})
assert sub.encounter_id.tolist()==ss.encounter_id.tolist() and sub.readmit_30d.notna().all()
sub.to_csv(f'{W}/submission_best.csv',index=False); print('saved',sub.shape,'range %.4f-%.4f mean %.4f'%(p.min(),p.max(),p.mean()))
from scipy.stats import spearmanr; print('rank corr with current submission.csv: %.4f'%spearmanr(p,cur.readmit_30d)[0])

### 13.2 Hill-climbing ensemble selection (greedy, nested)

In [ ]:
EXCLUDE = {'gam_ix','gam_ms4','woe_lr','woe_sum','gam_f6'}   # models trained after this step in the original timeline

In [ ]:
import numpy as np, pandas as pd, os, json, sys
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
W=WORK; D=DATA
te=pd.read_csv(f'{D}/test.csv'); ss=pd.read_csv(f'{D}/sample_submission.csv')
X=pd.read_parquet(f'{W}/features2.parquet',columns=['encounter_id'])
y=X.merge(pd.read_csv(f'{D}/train.csv'),on='encounter_id').readmit_30d.values
lg=lambda p:np.log(np.clip(p,1e-9,1-1e-9)/(1-np.clip(p,1e-9,1-1e-9)))
# ---- candidate pool: one entry per distinct model, no ablation/diagnostic runs ----
cand={}
def add(name,of,tf,is_prob=True):
    if not (os.path.exists(of) and os.path.exists(tf)): return
    o,t=np.load(of),np.load(tf)
    if len(o)!=len(y) or len(t)!=len(te) or np.isnan(o).any() or np.isnan(t).any(): print('skip',name); return
    cand[name]=(lg(o) if is_prob else o, lg(t) if is_prob else t)
cur=pd.read_csv(f'{W}/submission.csv'); np.save(f'{W}/_cur_test.npy',cur.readmit_30d.values)
add('old_blend',f'{W}/final_oof.npy',f'{W}/_cur_test.npy')
for n,f in [('old_xgb','xgb_oof_single'),('old_xgb_bag','xgb_oof_bag'),('old_cat','b_cat_oof'),('old_mlp','b_mlp_oof'),('rf','rf_oof'),('nb','nb_oof')]:
    add(n,f'{W}/{f}.npy',f'{W}/{f.replace("_oof","_test")}.npy')
for n in ['cat','lgb0','lgb1','lgb2','lr','xgb']: add('f3_'+n,f'{W}/n_{n}_oof.npy',f'{W}/n_{n}_test.npy')
for f in os.listdir(W):
    if f.startswith('v3_') and f.endswith('_oof.npy') and f[3:-8] not in EXCLUDE: add(f[3:-8],f'{W}/{f}',f'{W}/{f.replace("_oof","_test")}')
add('ridge',f'{W}/ls_ridge_a30000_oof.npy',f'{W}/ls_ridge_a30000_test.npy',is_prob=False)
add('linsvc',f'{W}/ls_linsvc_C0.0005_oof.npy',f'{W}/ls_linsvc_C0.0005_test.npy',is_prob=False)
names=sorted(cand); O=np.column_stack([cand[k][0] for k in names]); Ts=np.column_stack([cand[k][1] for k in names])
mu,sd=O.mean(0),O.std(0); O=(O-mu)/sd; Ts=(Ts-mu)/sd
print('candidates:',len(names))
for i,k in enumerate(names): print(f'  {k:18s} {roc_auc_score(y,O[:,i]):.5f}')
# ---- hill climbing with replacement (Caruana): greedy average, picks can repeat = integer weights ----
def climb(idx_tr, Omat, yy, iters=40, verbose=False):
    sel=[]; cursum=np.zeros(len(yy)); best_hist=[]
    for it in range(iters):
        bi,bs=None,-1
        for j in range(Omat.shape[1]):
            s=roc_auc_score(yy,(cursum+Omat[:,j])/(len(sel)+1))
            if s>bs: bs,bi=s,j
        sel.append(bi); cursum=cursum+Omat[:,bi]; best_hist.append(bs)
        if len(best_hist)>3 and bs<=max(best_hist[:-1])+1e-6 and it>=8: break
    return sel,best_hist
# honest estimate: run the whole selection inside each outer fold
outer=list(StratifiedKFold(5,shuffle=True,random_state=7).split(O,y))
oof_hc=np.zeros(len(y)); oof_st=np.zeros(len(y))
for f,(a,b) in enumerate(outer):
    sel,_=climb(a,O[a],y[a])
    w=np.bincount(sel,minlength=O.shape[1])/len(sel)
    oof_hc[b]=O[b]@w
    oof_st[b]=LogisticRegression(C=0.01).fit(O[a],y[a]).decision_function(O[b])
    print(f'outer {f}: hc {roc_auc_score(y[b],oof_hc[b]):.5f} | stack {roc_auc_score(y[b],oof_st[b]):.5f} | picks {[names[i] for i in dict.fromkeys(sel)][:6]}',flush=True)
print('NESTED-CV  hillclimb %.5f | logistic stack %.5f | best single %.5f'%(roc_auc_score(y,oof_hc),roc_auc_score(y,oof_st),max(roc_auc_score(y,O[:,i]) for i in range(len(names)))))
# ---- final fit on all training data ----
sel,hist=climb(np.arange(len(y)),O,y); w=np.bincount(sel,minlength=O.shape[1])/len(sel)
print('final hillclimb weights:',{names[i]:round(v,3) for i,v in enumerate(w) if v>0})
st=LogisticRegression(C=0.01).fit(O,y); 
res={'hc':(roc_auc_score(y,oof_hc),O@w,Ts@w),'stack':(roc_auc_score(y,oof_st),st.decision_function(O),st.decision_function(Ts))}
# 50/50 of the two combiners, judged by its own nested score
mix_o=0.5*((oof_hc-oof_hc.mean())/oof_hc.std())+0.5*((oof_st-oof_st.mean())/oof_st.std())
print('nested mix of both combiners %.5f'%roc_auc_score(y,mix_o))
json.dump({'names':names,'w':w.tolist()},open(f'{W}/hc_weights.json','w'))
np.save(f'{W}/hc_oof.npy',oof_hc); np.save(f'{W}/hc_test_z.npy',res['hc'][2]); np.save(f'{W}/st_oof.npy',oof_st); np.save(f'{W}/st_test_z.npy',res['stack'][2])

### 13.3 Stack robustness across fold seeds → `submission_v3_additive_stack.csv`

In [ ]:
EXCLUDE = {'gam_ix','gam_ms4','woe_lr','woe_sum','gam_f6'}

In [ ]:
import numpy as np, pandas as pd, os, json
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
W=WORK; D=DATA
X=pd.read_parquet(f'{W}/features2.parquet',columns=['encounter_id']); te=pd.read_csv(f'{D}/test.csv')
y=X.merge(pd.read_csv(f'{D}/train.csv'),on='encounter_id').readmit_30d.values
lg=lambda p:np.log(np.clip(p,1e-9,1-1e-9)/(1-np.clip(p,1e-9,1-1e-9)))
cand={}
def add(name,of,tf,is_prob=True):
    if not(os.path.exists(of) and os.path.exists(tf)): return
    o,t=np.load(of),np.load(tf)
    if len(o)!=len(y) or len(t)!=len(te) or np.isnan(o).any() or np.isnan(t).any(): return
    cand[name]=(lg(o) if is_prob else o, lg(t) if is_prob else t)
cur=pd.read_csv(f'{W}/submission.csv'); np.save(f'{W}/_cur_test.npy',cur.readmit_30d.values)
add('old_blend',f'{W}/final_oof.npy',f'{W}/_cur_test.npy')
for n,f in [('old_xgb','xgb_oof_single'),('old_xgb_bag','xgb_oof_bag'),('old_cat','b_cat_oof'),('old_mlp','b_mlp_oof'),('rf','rf_oof'),('nb','nb_oof')]:
    add(n,f'{W}/{f}.npy',f'{W}/{f.replace("_oof","_test")}.npy')
for n in ['cat','lgb0','lgb1','lgb2','lr','xgb']: add('f3_'+n,f'{W}/n_{n}_oof.npy',f'{W}/n_{n}_test.npy')
for f in sorted(os.listdir(W)):
    if f.startswith('v3_') and f.endswith('_oof.npy') and f[3:-8] not in EXCLUDE: add(f[3:-8],f'{W}/{f}',f'{W}/{f.replace("_oof","_test")}')
add('ridge',f'{W}/ls_ridge_a30000_oof.npy',f'{W}/ls_ridge_a30000_test.npy',False)
add('linsvc',f'{W}/ls_linsvc_C0.0005_oof.npy',f'{W}/ls_linsvc_C0.0005_test.npy',False)
names=sorted(cand); O=np.column_stack([cand[k][0] for k in names]); Ts=np.column_stack([cand[k][1] for k in names])
mu,sd=O.mean(0),O.std(0); O=(O-mu)/sd; Ts=(Ts-mu)/sd
ib=names.index('old_blend')
def nested(C,seed):
    o=np.zeros(len(y))
    for a,b in StratifiedKFold(5,shuffle=True,random_state=seed).split(O,y):
        o[b]=LogisticRegression(C=C).fit(O[a],y[a]).decision_function(O[b])
    return o
print('robustness of the stack across fold seeds and strength C:')
best=None
for C in [0.003,0.01,0.03,0.1]:
    s=[roc_auc_score(y,nested(C,sd_)) for sd_ in [0,7,13]]
    print(f'  C={C:<6} seeds {[round(v,5) for v in s]}  mean {np.mean(s):.5f}')
    if best is None or np.mean(s)>best[0]: best=(np.mean(s),C)
C=best[1]; print('chosen C',C,'mean nested AUC %.5f'%best[0])
o_st=nested(C,0); print('old_blend alone %.5f | stack %.5f'%(roc_auc_score(y,O[:,ib]),roc_auc_score(y,o_st)))
rng=np.random.RandomState(0); d=[]
for _ in range(400):
    i=rng.randint(0,len(y),len(y)); d.append(roc_auc_score(y[i],o_st[i])-roc_auc_score(y[i],O[i,ib]))
d=np.array(d); print('stack minus old_blend: %+.5f  95%% CI [%+.5f, %+.5f]  P(better)=%.2f'%(d.mean(),np.percentile(d,2.5),np.percentile(d,97.5),(d>0).mean()))
m=LogisticRegression(C=C).fit(O,y); print('weights:',{names[i]:round(w,3) for i,w in sorted(zip(range(len(names)),m.coef_[0]),key=lambda r:-abs(r[1])) if abs(w)>0.02})
z=m.decision_function(Ts); zo=m.decision_function(O)
pl=LogisticRegression(C=1e6).fit(zo.reshape(-1,1),y); p=pl.predict_proba(z.reshape(-1,1))[:,1]
ss=pd.read_csv(f'{D}/sample_submission.csv'); sub=pd.DataFrame({'encounter_id':te.encounter_id,'readmit_30d':np.round(p,6)})
assert sub.encounter_id.tolist()==ss.encounter_id.tolist() and sub.readmit_30d.notna().all() and sub.readmit_30d.between(0,1).all()
sub.to_csv(f'{W}/submission_v3_additive_stack.csv',index=False)
from scipy.stats import spearmanr
print('saved submission_v3_additive_stack.csv rows=%d range %.4f-%.4f mean %.4f'%(len(sub),p.min(),p.max(),p.mean()))
print('rank corr: vs submission.csv %.4f | vs submission_best.csv %.4f'%(spearmanr(p,cur.readmit_30d)[0],spearmanr(p,pd.read_csv(f'{W}/submission_best.csv').readmit_30d)[0]))

### 13.4 Non-negative hill-climb version → `submission_v3_hillclimb.csv`

In [ ]:
EXCLUDE = {'gam_ix','gam_ms4','woe_lr','woe_sum','gam_f6'}

In [ ]:
import numpy as np, pandas as pd, os
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from scipy.optimize import nnls
W=WORK; D=DATA
X=pd.read_parquet(f'{W}/features2.parquet',columns=['encounter_id']); te=pd.read_csv(f'{D}/test.csv'); ss=pd.read_csv(f'{D}/sample_submission.csv')
y=X.merge(pd.read_csv(f'{D}/train.csv'),on='encounter_id').readmit_30d.values
lg=lambda p:np.log(np.clip(p,1e-9,1-1e-9)/(1-np.clip(p,1e-9,1-1e-9)))
cand={}
def add(name,of,tf,is_prob=True):
    if not(os.path.exists(of) and os.path.exists(tf)): return
    o,t=np.load(of),np.load(tf)
    if len(o)!=len(y) or len(t)!=len(te) or np.isnan(o).any() or np.isnan(t).any(): return
    cand[name]=(lg(o) if is_prob else o, lg(t) if is_prob else t)
cur=pd.read_csv(f'{W}/submission.csv')
add('old_blend',f'{W}/final_oof.npy',f'{W}/_cur_test.npy')
for n,f in [('old_xgb','xgb_oof_single'),('old_xgb_bag','xgb_oof_bag'),('old_cat','b_cat_oof'),('old_mlp','b_mlp_oof'),('rf','rf_oof'),('nb','nb_oof')]:
    add(n,f'{W}/{f}.npy',f'{W}/{f.replace("_oof","_test")}.npy')
for n in ['cat','lgb0','lgb1','lgb2','lr','xgb']: add('f3_'+n,f'{W}/n_{n}_oof.npy',f'{W}/n_{n}_test.npy')
for f in sorted(os.listdir(W)):
    if f.startswith('v3_') and f.endswith('_oof.npy') and f[3:-8] not in EXCLUDE: add(f[3:-8],f'{W}/{f}',f'{W}/{f.replace("_oof","_test")}')
add('ridge',f'{W}/ls_ridge_a30000_oof.npy',f'{W}/ls_ridge_a30000_test.npy',False)
add('linsvc',f'{W}/ls_linsvc_C0.0005_oof.npy',f'{W}/ls_linsvc_C0.0005_test.npy',False)
names=sorted(cand); O=np.column_stack([cand[k][0] for k in names]); Ts=np.column_stack([cand[k][1] for k in names])
mu,sd=O.mean(0),O.std(0); O=(O-mu)/sd; Ts=(Ts-mu)/sd; ib=names.index('old_blend')
def climb(Om,yy,iters=40):
    sel=[]; cs=np.zeros(len(yy)); hist=[]
    for it in range(iters):
        bi,bs=None,-1
        for j in range(Om.shape[1]):
            s=roc_auc_score(yy,(cs+Om[:,j])/(len(sel)+1))
            if s>bs: bs,bi=s,j
        sel.append(bi); cs=cs+Om[:,bi]; hist.append(bs)
        if len(hist)>3 and bs<=max(hist[:-1])+1e-6 and it>=8: break
    return np.bincount(sel,minlength=Om.shape[1])/len(sel)
def nested(kind,seed,C=0.03):
    o=np.zeros(len(y))
    for a,b in StratifiedKFold(5,shuffle=True,random_state=seed).split(O,y):
        if kind=='stack': o[b]=LogisticRegression(C=C).fit(O[a],y[a]).decision_function(O[b])
        else: o[b]=O[b]@climb(O[a],y[a])
    return o
sc={}
for kind in ['stack','hc']:
    s=[roc_auc_score(y,nested(kind,sd_)) for sd_ in [0,7,13]]; sc[kind]=np.mean(s); print(f'{kind:6s} nested across seeds {[round(v,5) for v in s]} mean {np.mean(s):.5f}',flush=True)
print('old_blend alone %.5f'%roc_auc_score(y,O[:,ib]))
o_st=nested('stack',0); o_hc=nested('hc',0)
rng=np.random.RandomState(0); dh=[];ds=[];dd=[]
for _ in range(400):
    i=rng.randint(0,len(y),len(y)); base=roc_auc_score(y[i],O[i,ib])
    a1=roc_auc_score(y[i],o_st[i]); a2=roc_auc_score(y[i],o_hc[i]); ds.append(a1-base); dh.append(a2-base); dd.append(a1-a2)
for lbl,d in [('stack - old',ds),('hillclimb - old',dh),('stack - hillclimb',dd)]:
    d=np.array(d); print('%-20s %+.5f  95%% CI [%+.5f, %+.5f]  P(>0)=%.2f'%(lbl,d.mean(),np.percentile(d,2.5),np.percentile(d,97.5),(d>0).mean()))
w=climb(O,y); print('\nhillclimb weights (all non-negative):',{names[i]:round(v,3) for i,v in enumerate(w) if v>0})
zo=O@w; zt=Ts@w
pl=LogisticRegression(C=1e6).fit(zo.reshape(-1,1),y); p=pl.predict_proba(zt.reshape(-1,1))[:,1]
sub=pd.DataFrame({'encounter_id':te.encounter_id,'readmit_30d':np.round(p,6)})
assert sub.encounter_id.tolist()==ss.encounter_id.tolist() and sub.readmit_30d.notna().all() and sub.readmit_30d.between(0,1).all()
sub.to_csv(f'{W}/submission_v3_hillclimb.csv',index=False)
from scipy.stats import spearmanr
print('saved submission_v3_hillclimb.csv rows=%d range %.4f-%.4f mean %.4f | rank corr vs v3 stack %.4f'%(len(sub),p.min(),p.max(),p.mean(),spearmanr(p,pd.read_csv(f'{W}/submission_v3_additive_stack.csv').readmit_30d)[0]))

### 13.5 Add the notebook-12 models and re-stack → `submission_v4_stack.csv`

In [ ]:
EXCLUDE = {'gam_f6'}   # gam_f6 did not exist when v4 was built

In [ ]:
import numpy as np, pandas as pd, os
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
W=WORK; D=DATA
X=pd.read_parquet(f'{W}/features2.parquet',columns=['encounter_id']); te=pd.read_csv(f'{D}/test.csv'); ss=pd.read_csv(f'{D}/sample_submission.csv')
y=X.merge(pd.read_csv(f'{D}/train.csv'),on='encounter_id').readmit_30d.values
lg=lambda p:np.log(np.clip(p,1e-9,1-1e-9)/(1-np.clip(p,1e-9,1-1e-9)))
cand={}
def add(name,of,tf,is_prob=True):
    if not(os.path.exists(of) and os.path.exists(tf)): return
    o,t=np.load(of),np.load(tf)
    if len(o)!=len(y) or len(t)!=len(te) or np.isnan(o).any() or np.isnan(t).any(): print('skip',name); return
    cand[name]=(lg(o) if is_prob else o, lg(t) if is_prob else t)
add('old_blend',f'{W}/final_oof.npy',f'{W}/_cur_test.npy')
for n,f in [('old_xgb','xgb_oof_single'),('old_xgb_bag','xgb_oof_bag'),('old_cat','b_cat_oof'),('old_mlp','b_mlp_oof'),('rf','rf_oof'),('nb','nb_oof')]:
    add(n,f'{W}/{f}.npy',f'{W}/{f.replace("_oof","_test")}.npy')
for n in ['cat','lgb0','lgb1','lgb2','lr','xgb']: add('f3_'+n,f'{W}/n_{n}_oof.npy',f'{W}/n_{n}_test.npy')
for f in sorted(os.listdir(W)):
    if f.startswith('v3_') and f.endswith('_oof.npy') and f[3:-8] not in EXCLUDE: add(f[3:-8],f'{W}/{f}',f'{W}/{f.replace("_oof","_test")}')
add('ridge',f'{W}/ls_ridge_a30000_oof.npy',f'{W}/ls_ridge_a30000_test.npy',False)
add('linsvc',f'{W}/ls_linsvc_C0.0005_oof.npy',f'{W}/ls_linsvc_C0.0005_test.npy',False)
NEW={'gam_ix','gam_ms4','woe_lr','woe_sum'}
names=sorted(cand); O=np.column_stack([cand[k][0] for k in names]); Ts=np.column_stack([cand[k][1] for k in names])
mu,sd=O.mean(0),O.std(0); O=(O-mu)/sd; Ts=(Ts-mu)/sd
oldi=[i for i,k in enumerate(names) if k not in NEW]; alli=list(range(len(names)))
print('pool A (previous) =',len(oldi),'| pool B (with new) =',len(alli))
for k in sorted(NEW):
    if k in names: print(f'  new candidate {k:12s} {roc_auc_score(y,O[:,names.index(k)]):.5f}')
def nested(idx,C,seed):
    o=np.zeros(len(y))
    for a,b in StratifiedKFold(5,shuffle=True,random_state=seed).split(O,y):
        o[b]=LogisticRegression(C=C).fit(O[a][:,idx],y[a]).decision_function(O[b][:,idx])
    return o
print('\nnested-CV AUC (mean over fold seeds 0/7/13):')
best={}
for lbl,idx in [('A previous pool',oldi),('B with new models',alli)]:
    for C in [0.01,0.03,0.1]:
        s=[roc_auc_score(y,nested(idx,C,sd_)) for sd_ in [0,7,13]]
        print(f'  {lbl:18s} C={C:<5} {[round(v,5) for v in s]} mean {np.mean(s):.5f}')
        if lbl not in best or np.mean(s)>best[lbl][0]: best[lbl]=(np.mean(s),C)
print(best)
oA=nested(oldi,best['A previous pool'][1],0); oB=nested(alli,best['B with new models'][1],0)
rng=np.random.RandomState(0); d=[]
for _ in range(400):
    i=rng.randint(0,len(y),len(y)); d.append(roc_auc_score(y[i],oB[i])-roc_auc_score(y[i],oA[i]))
d=np.array(d); print('\nB minus A: %+.5f  95%% CI [%+.5f, %+.5f]  P(B better)=%.2f'%(d.mean(),np.percentile(d,2.5),np.percentile(d,97.5),(d>0).mean()))
if d.mean()>0:
    C=best['B with new models'][1]; m=LogisticRegression(C=C).fit(O[:,alli],y)
    print('weights:',{names[alli[i]]:round(w,3) for i,w in sorted(enumerate(m.coef_[0]),key=lambda r:-abs(r[1])) if abs(w)>0.03})
    zo=m.decision_function(O[:,alli]); zt=m.decision_function(Ts[:,alli])
    pl=LogisticRegression(C=1e6).fit(zo.reshape(-1,1),y); p=pl.predict_proba(zt.reshape(-1,1))[:,1]
    sub=pd.DataFrame({'encounter_id':te.encounter_id,'readmit_30d':np.round(p,6)})
    assert sub.encounter_id.tolist()==ss.encounter_id.tolist() and sub.readmit_30d.notna().all() and sub.readmit_30d.between(0,1).all()
    sub.to_csv(f'{W}/submission_v4_stack.csv',index=False)
    from scipy.stats import spearmanr
    prev=pd.read_csv(f'{W}/submission_v3_additive_stack.csv').readmit_30d
    print('saved submission_v4_stack.csv rows=%d range %.4f-%.4f mean %.4f | rank corr vs v3 %.4f'%(len(sub),p.min(),p.max(),p.mean(),spearmanr(p,prev)[0]))
else: print('new models do not help; keeping v3 file')

In [ ]:
import pandas as pd
def validate_submission(path):
    """Checks a file against sample_submission.csv: same ids in the same order, no gaps, probabilities in [0, 1]."""
    s = pd.read_csv(path); ss = pd.read_csv(f'{DATA}/sample_submission.csv')
    ok = s.encounter_id.tolist() == ss.encounter_id.tolist()
    print(Path(path).name, '| rows', len(s), '| ids match sample:', ok, '| NaN:', int(s.readmit_30d.isna().sum()),
          '| all in [0,1]:', bool(s.readmit_30d.between(0, 1).all()), '| mean %.4f' % s.readmit_30d.mean())
    assert ok and s.readmit_30d.notna().all() and s.readmit_30d.between(0, 1).all()

In [ ]:
for f in ['submission_best.csv','submission_v3_additive_stack.csv','submission_v3_hillclimb.csv','submission_v4_stack.csv']:
    validate_submission(f'{WORK}/{f}')